# Lab 0-03: One LLM, an API Request, and Structured Output

Use this tutorial before `03_model_comparison.ipynb`. You will use the one model configured in this lab's `.env` twice on the same synthetic forensic task. First, you will inspect a normal chat response. Then, you will ask for a response that follows a JSON Schema.

The goal is to understand the request-and-response pattern before comparing three different models. Structured output makes a response easier for a program to read because the program knows which fields to expect.

## Step 0: Load This Lab's Settings

Run this cell from the `lab0_03_model_basics` folder. It reads the configured `MODEL` and `OLLAMA_BASE_URL`, then creates the same OpenAI-compatible client used in the later labs.

In [ ]:
import json
from pathlib import Path

from dotenv import dotenv_values
from openai import OpenAI

LAB_NAME = 'lab0_03_model_basics'
lab_dir = Path.cwd().resolve()
if lab_dir.name != LAB_NAME:
    raise FileNotFoundError(f'Open this notebook from the {LAB_NAME} folder.')

env_path = lab_dir / '.env'
if not env_path.exists():
    raise FileNotFoundError('Expected .env in this folder. Copy .env.example to .env first.')

config = dotenv_values(env_path)
model = config.get('MODEL')
ollama_base_url = config.get('OLLAMA_BASE_URL')
if not model or not ollama_base_url:
    raise ValueError("MODEL or OLLAMA_BASE_URL is missing from this lab's .env")

client = OpenAI(base_url=ollama_base_url, api_key='ollama')
print('Model:', model)
print('Ollama address:', ollama_base_url)

## Step 1: Ask One Ordinary API Question

A chat-completions request has a model name and a list of messages. This first request uses no output constraint, so the model is free to choose the wording and layout of its answer.

In [ ]:
case_note = '''
Investigator Maya Chen documented an interview with Jordan Lee.
Jordan said a suspicious text came from 415-555-0187.
The seized phone record listed IMEI 356938035643809.
'''.strip()

task_prompt = f'''
Read this synthetic case note. Identify the investigator, the interviewed person,
the suspicious phone number, and the device ID. Give a brief case summary.

Case note:
{case_note}
'''.strip()

plain_response = client.chat.completions.create(
    model=model,
    messages=[{'role': 'user', 'content': task_prompt}],
)
plain_text = plain_response.choices[0].message.content
print(plain_text)

The answer may be useful to a person, but its layout is chosen by the model. A program that needs exact fields would need to guess where each fact appears.

## Step 2: Define the Output Shape

A JSON Schema describes the fields and value types the program expects. The schema below requires a short summary plus four extraction fields.

In [ ]:
extraction_schema = {
    'type': 'object',
    'properties': {
        'case_summary': {'type': 'string'},
        'investigator': {'type': 'string'},
        'interviewed_person': {'type': 'string'},
        'suspicious_phone_number': {'type': 'string'},
        'device_id': {'type': 'string'},
    },
    'required': [
        'case_summary', 'investigator', 'interviewed_person',
        'suspicious_phone_number', 'device_id',
    ],
    'additionalProperties': False,
}

print(json.dumps(extraction_schema, indent=2))

## Step 3: Request Structured Output

This request uses the same model and case note, but adds `response_format`. Keep the prompt specific as well: the schema controls the shape, while the prompt tells the model what the values should mean.

In [ ]:
structured_response = client.chat.completions.create(
    model=model,
    messages=[{'role': 'user', 'content': task_prompt}],
    response_format={
        'type': 'json_schema',
        'json_schema': {
            'name': 'forensic_case_extraction',
            'strict': True,
            'schema': extraction_schema,
        },
    },
)
structured_text = structured_response.choices[0].message.content
print(structured_text)

## Step 4: Parse and Check the Result

JSON text is still text until your program parses it. This check confirms that every required field is present and contains a string. In a larger application, the same checks would protect later steps that depend on the extraction.

In [ ]:
structured_data = json.loads(structured_text)
required_fields = extraction_schema['required']
missing_fields = [field for field in required_fields if field not in structured_data]
non_string_fields = [
    field for field in required_fields
    if field in structured_data and not isinstance(structured_data[field], str)
]
unexpected_fields = sorted(set(structured_data) - set(extraction_schema['properties']))

if missing_fields or non_string_fields or unexpected_fields:
    raise ValueError({
        'missing_fields': missing_fields,
        'non_string_fields': non_string_fields,
        'unexpected_fields': unexpected_fields,
    })

print('Structured output passed the checks.')
print(json.dumps(structured_data, indent=2))

## Step 5: Short Exercise—Add One Field

Add a required `recommended_next_step` string field to both `properties` and `required`. Then update `task_prompt` so the model recommends one human review step based only on the case note. Rerun Steps 3 and 4.

Your result is complete when the parsed output includes `recommended_next_step` and the validation check still passes.

## Before You Continue

You have now used one LLM through an API and turned a model response into a predictable data object. Next, open `03_model_comparison.ipynb`, where three models will receive the same prompt and you will compare their outputs.